In [23]:
import GridMLM_tokenizers
from GridMLM_tokenizers import CSGridMLMTokenizer
import numpy as np
from graph_utils import make_bar_objects, MelodicHarmonization
from models_graph import HarmonicGraphEncoder
from models_BiLSTM import HarmonyBiLSTM
import torch
import ast

In [4]:
tokenizer = CSGridMLMTokenizer(
    fixed_length=80,
    quantization='4th',
    intertwine_bar_info=True,
    trim_start=False,
    use_pc_roll=True,
    use_full_range_melody=False
)

In [5]:
chord_features = GridMLM_tokenizers.CHORD_FEATURES
chord_id_features = {tokenizer.vocab[k]: v for k, v in chord_features.items()}

In [6]:
print(chord_features.keys())
print(chord_features.values())

dict_keys(['C:maj', 'C#:maj', 'D:maj', 'D#:maj', 'E:maj', 'F:maj', 'F#:maj', 'G:maj', 'G#:maj', 'A:maj', 'A#:maj', 'B:maj', 'C:min', 'C#:min', 'D:min', 'D#:min', 'E:min', 'F:min', 'F#:min', 'G:min', 'G#:min', 'A:min', 'A#:min', 'B:min', 'C:aug', 'C#:aug', 'D:aug', 'D#:aug', 'E:aug', 'F:aug', 'F#:aug', 'G:aug', 'G#:aug', 'A:aug', 'A#:aug', 'B:aug', 'C:dim', 'C#:dim', 'D:dim', 'D#:dim', 'E:dim', 'F:dim', 'F#:dim', 'G:dim', 'G#:dim', 'A:dim', 'A#:dim', 'B:dim', 'C:sus4', 'C#:sus4', 'D:sus4', 'D#:sus4', 'E:sus4', 'F:sus4', 'F#:sus4', 'G:sus4', 'G#:sus4', 'A:sus4', 'A#:sus4', 'B:sus4', 'C:sus2', 'C#:sus2', 'D:sus2', 'D#:sus2', 'E:sus2', 'F:sus2', 'F#:sus2', 'G:sus2', 'G#:sus2', 'A:sus2', 'A#:sus2', 'B:sus2', 'C:7', 'C#:7', 'D:7', 'D#:7', 'E:7', 'F:7', 'F#:7', 'G:7', 'G#:7', 'A:7', 'A#:7', 'B:7', 'C:maj7', 'C#:maj7', 'D:maj7', 'D#:maj7', 'E:maj7', 'F:maj7', 'F#:maj7', 'G:maj7', 'G#:maj7', 'A:maj7', 'A#:maj7', 'B:maj7', 'C:min7', 'C#:min7', 'D:min7', 'D#:min7', 'E:min7', 'F:min7', 'F#:min7', 

In [7]:
s = '[0,4,7]'
n = ast.literal_eval(s)
print(n)

[0, 4, 7]


### String format

- Each bar starts with the prefix `b_`.
- Chord segments are separated by `;`.
- Each chord segment includes information about a) chord symbol; b) position in bar; and c) melody pitches.
- - Chord symbols are described by their `mir_eval` symbol (`ROOT:TYPE`). E.g., `C:maj`.
- - Bar position is prefixed with `_@`. E.g., `_@0`.
- - Melody pitches are prefixed with `_m` and then pitches are given in a list-like string forma. E.g., `_m[0,4,7,9]`.

### Complete example

`in_seq = 'b_C:maj_@0_m[0,4,9];A:7_@2_m[1,4]b_D:min_@0_m[2,5,7];G:7_@2_m[5,9,11]'`

### Incomplete input

Bar position and/or melody information can be ommitted. For example, for a quick run, such strings are also valid:

`in_seq = 'C:maj;A:7;D:min;G:7'`

`in_seq = 'b_C:maj;A:7_@2b_D:min;G:7'`

`in_seq = 'b_C:maj_@0;A:7_@2b_D:min_@0;G:7_@2'`

In [8]:
# in_seq = 'b_C:maj_@0_m[0,4,9];A:7_@2_m[1,4]b_D:min_@0_m[2,5,7];G:7_@2_m[5,9,11]'
# in_seq = 'C:maj;A:7;D:min;G:7'
# in_seq = 'b_C:maj;A:7_@2b_D:min;G:7'
in_seq = 'b_C:maj_@0;A:7_@2b_D:min_@0;G:7_@2'

In [9]:
if 'b_' in in_seq:
    bar_split = in_seq.split('b_')[1:]
else:
    bar_split = [in_seq]

In [10]:
bars = []

for bs in bar_split:
    chords_split = bs.split(';')
    bar_chord_ids = []
    bar_position_info = []
    bar_melody_info = []
    position_info = 0
    bar_token_positions_info = []
    bar_idx = 0
    for i, cs in enumerate(chords_split):
        melody_info = []
        if '_@' in cs:
            position_split = cs.split('_@')
            chord_symbol = position_split[0]
            if '_m' in position_split[1]:
                melody_split = position_split[1].split('_m')
                position_info = int(melody_split[0])
                melody_info = ast.literal_eval(melody_split[1])
        elif '_m' in cs:
            melody_split = position_split[1].split['_m']
            chord_symbol = melody_split[0]
            melody_info = ast.literal_eval(melody_split[1])
        else:
            chord_symbol = cs
        if chord_symbol in tokenizer.vocab.keys():
            print(f'{chord_symbol} in vocab as: {tokenizer.vocab[chord_symbol]}')
            bar_chord_ids.append(tokenizer.vocab[chord_symbol])
            bar_position_info.append(position_info)
            bar_token_positions_info.append(position_info + bar_idx)
            if '_@' not in cs:
                position_info += 2
            bar_melody_info.append(melody_info)
        else:
            print(f'unrecognized chord symbol {chord_symbol}')
    bar_idx += 1
    current_bar = {
        'chord_ids': bar_chord_ids,
        'melody_pcs': bar_melody_info,
        'chord_token_positions': bar_token_positions_info,
        'bar_token_positions': bar_position_info
    }
    bars.append(current_bar)

C:maj in vocab as: 7
A:7 in vocab as: 274
D:min in vocab as: 66
G:7 in vocab as: 216


In [11]:
bar_objects = make_bar_objects(bars)

In [12]:
print(bar_objects[0].print_info())

Bar token positions: [0, 0]
Number of chord objects in bar: 2
Chord object 1:
Chord label: C:maj
Pitch classes: [0, 4, 7]
Root: 0
Chord ID: 7
Bar Positions: [0]
Token Positions: [0]
Melody PCs: [[]]
Graph Features:
tensor([[1., 0., 0., 0., 0., 1., 0., 0.],
        [0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.]])
BiLSTM Features
tensor([1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.])
Chord object 2:
Chord label: A:7
Pitch classes: [1, 4, 7, 9]
Root: 9
Chord ID: 274
Bar Positions: [1]
Token Positions: [0]
Melody PCs: [[]]
Graph Features:
tensor([[0., 1., 0., 0., 0., 1., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0., 1., 0., 0.],
        [1., 0., 0., 0., 0., 1., 0., 0.]])
BiLSTM Features
tensor([0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0.])
None


In [13]:
m = MelodicHarmonization(bar_objects)

In [14]:
m.make_graph_of_segment(0,len(bar_objects))
m.make_bilstm_seq_of_segment(0,len(bar_objects))

In [15]:
graph_model = HarmonicGraphEncoder()
bilstm_model = HarmonyBiLSTM()

In [ ]:
y_graph = graph_model(m.segment_graph)

In [19]:
print(y_graph.shape)

torch.Size([128])


In [27]:
print(m.segment_bilstm)
y_bilstm = bilstm_model(m.segment_bilstm.unsqueeze(0), torch.tensor([m.segment_bilstm.shape[0]]))

tensor([[1., 0., 0., 0., 1., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 1., 0., 1., 0., 0., 0., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.]])


In [28]:
print(y_bilstm.shape)

torch.Size([1, 128])
